# Day 59: Reliability, Responsible AI and Demo Prep

**Today's question:** what could go wrong, in front of an audience or for a real user, and how do you find it
**before** they do?

Five tools: a failure-case table, two hallucination checks, redaction, bias probes, and an offline switch for
the demo. Before running a cell marked 🤔, predict what it will show.

## 1. A failure-case table

List inputs and situations that might break the app, run each one, and write down **what happens now** next to
**what should happen**. Here is the core of our Spending Summary app (from Day 57), as one function that
returns the text `main()` would print.

In [ ]:
import csv
from pathlib import Path

expenses = []
with open(Path("../../../datasets/expenses.csv"), encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        expenses.append({"date": row["date"], "category": row["category"], "amount": float(row["amount"])})


def summarise_month(month):
    """What the app prints for one month (core only, no AI)."""
    month_expenses = [e for e in expenses if e["date"].startswith(month)]
    if not month_expenses:
        return f"No expenses found for '{month}'."
    total = sum([e["amount"] for e in month_expenses])
    return f"{len(month_expenses)} expenses, total {total:.2f}"


print(summarise_month("2026-09"))

🤔 **Predict** what each input does before running the table:

In [ ]:
cases = ["2026-09", "", "2025-01", "september", "2026-9", "  2026-10  "]
for month in cases:
    shown = f"'{month}'"  # quotes make the empty input and the spaces visible
    print(f"{shown:<14} -> {summarise_month(month)}")

The empty input is a **real bug**: pressing Enter summarises *all three months* (every date "starts with" the
empty string). The Day 57 and Day 58 `example_app` versions have this bug too. Nobody noticed, because nobody
tried it. That is exactly what the table is for.

| Input / situation | What happens now | What should happen | Status |
|-------------------|------------------|--------------------|--------|
| `2026-09` | 49 expenses | same | ok |
| empty input | all 148 expenses | ask again | **bug** |
| `2025-01` | "No expenses found" | same | ok |
| `september`, `2026-9` | "No expenses found" | explain the format `YYYY-MM` | improve |
| `  2026-10  ` | "No expenses found" | strip spaces, then 52 expenses | improve |
| model reply not JSON / refusal | template fallback | same | ok (tested Day 58) |
| no key / no network | mock answers | same | ok (Wi-Fi-off rehearsal below) |

In [ ]:
def is_valid_month(text):
    """True for text like '2026-09': 4 digits, a dash, 2 digits from 01 to 12."""
    if len(text) != 7 or text[4] != "-":
        return False
    year, month = text[:4], text[5:]
    return year.isdigit() and month.isdigit() and 1 <= int(month) <= 12


for month in cases:
    shown = f"'{month}'"
    cleaned = month.strip()
    if is_valid_month(cleaned):
        print(f"{shown:<14} -> {summarise_month(cleaned)}")
    else:
        print(f"{shown:<14} -> Please type a month like 2026-09.")

Every row of the table becomes a test: `assert is_valid_month("") is False`, `assert is_valid_month("2026-13") is False`.

## 2. Hallucination check 1: numbers must come from the data

Our prompt says "only use numbers that appear above", but an instruction is not a guarantee. Code can check it:
every number in the summary must be one of the totals (or their sum).

In [ ]:
def numbers_in(text):
    """All numbers written in a text, as floats: 'spent 12.50, mostly food.' -> [12.5]"""
    numbers = []
    for word in text.split():
        word = word.strip(".,;:!?()%")  # remove these characters from both ends
        try:
            numbers.append(float(word))
        except ValueError:
            continue
    return numbers


def invented_numbers(summary, totals):
    """Numbers in the summary that are not a category total or the grand total."""
    allowed = [round(amount, 2) for amount in totals.values()]
    allowed.append(round(sum(totals.values()), 2))
    return [number for number in numbers_in(summary) if round(number, 2) not in allowed]


september = {"food": 660.87, "rent": 650.0, "transport": 187.37, "entertainment": 156.61,
             "shopping": 108.56, "health": 101.59, "utilities": 64.2}
print(numbers_in("You spent 1929.20 in total; food was 660.87."))

Two **hand-written** summaries (examples, not real model output). 🤔 **Predict** which numbers get flagged:

In [ ]:
grounded = "You spent 1929.20 in total. Your biggest flexible category was food at 660.87."
invented = "You spent 1929.20 in total, 12% more than August, and food was 700.00."

print(invented_numbers(grounded, september))
print(invented_numbers(invented, september))

`12` (we never gave the model August) and `700.0` (food was 660.87) are flagged. In `write_summary` this is one
extra check after parsing: *if `invented_numbers(...)` is not empty, use the template instead.* Apply it to the
summary only; a tip such as "save 20 per week" is advice, not a claim about the data.

## 3. Hallucination check 2: the answer must quote its source

For a Q&A assistant (option C) we ask the model for its answer **plus** a short quote copied from the FAQ section it
used. We cannot check that the meaning is right, but we can check that the evidence really exists.

In [ ]:
faq_sections = {
    "S02": "Students can borrow up to 6 books at a time for 14 days. Each book can be renewed twice online.",
    "S03": "The late fee is 0.50 per book per day, up to a maximum of 10.00 per book.",
}


def quote_is_supported(quote, section_text):
    """True if the quote (ignoring case and a final full stop) appears in the section."""
    cleaned = quote.strip().strip(".").lower()
    return cleaned != "" and cleaned in section_text.lower()


replies = [  # hand-written examples of model replies, already parsed from JSON
    {"answer": "You can borrow 6 books.", "section_id": "S02", "quote": "borrow up to 6 books at a time"},
    {"answer": "You can borrow 8 books.", "section_id": "S02", "quote": "borrow up to 8 books"},
    {"answer": "The fee is 0.50 a day.", "section_id": "S07", "quote": "0.50 per book per day"},
]

🤔 **Predict** which replies pass. Careful with the third one.

In [ ]:
for reply in replies:
    section_text = faq_sections.get(reply["section_id"], "")
    print(reply["section_id"], quote_is_supported(reply["quote"], section_text), "|", reply["answer"])

The second reply invented "8"; the third cites a section (`S07`) we never gave it. Both are rejected, and the app
shows the best FAQ section word for word instead. The option C reference uses exactly this check.

## 4. Privacy: redact before sending

Rule 1 is still Day 56's: **do not send what you do not need.** Rule 2: when you must send user text, remove
contact details first. `reviews.csv` contains one review with (fictional) contact details:

In [ ]:
import pandas as pd

reviews = pd.read_csv(Path("../../../datasets/reviews.csv"))
r33 = reviews[reviews["review_id"] == "R33"]["review_text"].iloc[0]
print(r33)

In [ ]:
def redact(text):
    """Replace e-mail addresses and phone-like numbers (7+ digits) with placeholders."""
    clean_words = []
    for word in text.split():
        bare = word.strip(".,;:!?()")  # remove punctuation from both ends
        digit_count = len([char for char in bare if char.isdigit()])
        if "@" in bare and "." in bare:
            clean_words.append("[email]")
        elif digit_count >= 7:
            clean_words.append("[number]")
        else:
            clean_words.append(word)
    return " ".join(clean_words)


print(redact(r33))

🤔 **Predict:** what does `redact` miss in *"I'm Jo Smith, call 555 0142"*?

In [ ]:
print(redact("I'm Jo Smith, call 555 0142"))

Both: the **name** (no simple rule can find names reliably) and the phone number written with a space (two
short words of digits). Redaction **reduces** risk; it does not remove it. Write that in your README's
limitations, and tell users not to type personal details.

## 5. Bias questions

Ask of every capstone: *Who is in the data, and who is missing? Does the output change for a reason that should
not matter?* A simple probe is a **counterfactual test**: change one thing that should not matter, keep
everything else the same, and compare.

In [ ]:
students = pd.read_csv(Path("../../../datasets/students.csv"))
print(students["has_job"].value_counts())
students.groupby("has_job")["passed"].mean().round(2)  # share of students who passed

Students with a job pass less often in this (synthetic) data. Suppose we train a model **with** `has_job` as a
feature:

In [ ]:
from sklearn.linear_model import LogisticRegression

students["job"] = (students["has_job"] == "yes").astype(int)  # yes -> 1, no -> 0
features = ["study_hours", "attendance_pct", "previous_score", "job"]
model = LogisticRegression(max_iter=1000)
model.fit(students[features], students["passed"])

same_habits = pd.DataFrame([[8, 80, 60, 0], [8, 80, 60, 1]], columns=features)
chances = model.predict_proba(same_habits)[:, 1]  # column 1 = chance of passing
print(f"no job: {chances[0]:.0%}   job: {chances[1]:.0%}")

🤔 **Discuss:** two students with **identical study habits**; one has a part-time job and gets a much lower
predicted chance. Is that a fair thing for an app to say? What if a school used it to decide who gets a
scholarship?

The pattern may be real in the data, and still unfair to act on: it would penalise students who *need* to work.
The option A reference therefore leaves out `has_job` and `program` and uses only habits a student can change,
and says so in its README. For an LLM app, run the same probe by swapping names or genders in a test input and
comparing the outputs.

## 6. The offline switch for your demo

Networks fail in classrooms. Rehearse your demo **in offline mode** so that you know exactly what the mock
shows, and switch to it in one command if the API or Wi-Fi fails.

In [ ]:
import os


def llm_mode():
    """Which model will answer right now?"""
    if os.environ.get("COURSE_OFFLINE") == "1" or not os.environ.get("ANTHROPIC_API_KEY"):
        return "offline mock"
    return "online"


print("Mode now:", llm_mode())
os.environ["COURSE_OFFLINE"] = "1"
print("After setting COURSE_OFFLINE=1:", llm_mode())

In the terminal, for one run only:

```
COURSE_OFFLINE=1 uv run python main.py                         # macOS / Linux
$env:COURSE_OFFLINE="1"; uv run python main.py                  # Windows PowerShell
```

Print the mode at start-up (`AI mode: offline mock`) so the audience always knows what they are seeing.

## 7. Common mistake (this cell fails on purpose)

`numbers_in` uses `try/except` because text is messy. Without it, the first non-number crashes:

In [ ]:
float("1,929.20")

Note the side effect: `numbers_in` silently **skips** "1,929.20", so a model that writes numbers with thousands
separators slips past the check. Every check has limits; write them down.

| Mistake | Symptom | Fix |
|---------|---------|-----|
| testing only the "happy path" | demo crashes on Enter or a typo | failure-case table; every row becomes a test |
| trusting "only use the data" in the prompt | invented numbers or sources | check numbers / quotes in code, fall back |
| sending raw user text | contact details leave the computer | minimise, then `redact`, and warn users |
| "the data says so, so it's fair" | the app penalises a group for a reason that should not matter | counterfactual probe; drop or justify the feature |
| live demo needs Wi-Fi | demo dies | rehearse with `COURSE_OFFLINE=1`; mode shown at start-up |

## 8. README and demo script

Your README is how a stranger (or the grader) understands and runs your project. Use these headings:

```
# <Project name>
One sentence: what it does and for whom.
## How to run         (offline: no key needed; online: .env with ANTHROPIC_API_KEY; tests: uv run pytest -q)
## Example run        (a real transcript, copied from your terminal, mode shown)
## How it works       (data -> core -> the one AI function -> output; 5 lines)
## Evaluation         (your success criteria and the numbers you measured, with date and mode/model)
## Responsible AI     (limitations · privacy: what is sent, what is redacted · bias · when NOT to trust it)
## What I would do next
```

**5-minute demo script** (write it, then rehearse it once with a timer):

| Time | What you show |
|------|---------------|
| 0:00–0:30 | the problem and the user, in one or two sentences |
| 0:30–3:00 | live run: one normal case, one edge case, one failure handled (bad input or bad reply) |
| 3:00–4:00 | how it works: the one AI function and one test you are proud of |
| 4:00–5:00 | one limitation, one privacy/bias point, what you would do next |
| backup | Wi-Fi or API fails → `COURSE_OFFLINE=1` run; still fails → README transcript |

## Recap

1. A failure-case table finds the bugs nobody tries; each row becomes a test.
2. Check model claims in code: numbers must come from the data, quotes must exist in the source.
3. Privacy: send the minimum, redact contact details, admit what redaction misses.
4. Bias: who is (not) in the data; counterfactual probes; drop features that should not matter.
5. README with a Responsible AI section, and a rehearsed 5-minute demo with an offline backup.

**Next time:** final presentations, the final assessment, and what to learn next.